In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.T
    S_scaled = S / np.sqrt(d_k)
    A = softmax(S_scaled)
    Output = A @ V
    return Output, A

In [ ]:
np.random.seed(42)
seq_len = 4
d_model = 8
num_heads = 2
d_k = 4
d_v = 4

X = np.random.randn(seq_len, d_model)
print("X shape:", X.shape)

Wq_1 = np.random.randn(d_model, d_k) * 0.1
Wk_1 = np.random.randn(d_model, d_k) * 0.1
Wv_1 = np.random.randn(d_model, d_v) * 0.1

Wq_2 = np.random.randn(d_model, d_k) * 0.1
Wk_2 = np.random.randn(d_model, d_k) * 0.1
Wv_2 = np.random.randn(d_model, d_v) * 0.1

Q1 = X @ Wq_1
K1 = X @ Wk_1
V1 = X @ Wv_1

Q2 = X @ Wq_2
K2 = X @ Wk_2
V2 = X @ Wv_2

print("Q1 shape:", Q1.shape, "K1 shape:", K1.shape, "V1 shape:", V1.shape)
print("Q2 shape:", Q2.shape, "K2 shape:", K2.shape, "V2 shape:", V2.shape)

head1, A1 = scaled_dot_product_attention(Q1, K1, V1)
head2, A2 = scaled_dot_product_attention(Q2, K2, V2)

print("head1 shape:", head1.shape, "A1 shape:", A1.shape)
print("head2 shape:", head2.shape, "A2 shape:", A2.shape)

concat = np.concatenate([head1, head2], axis=-1)
print("Concatenated shape:", concat.shape)

Wo = np.random.randn(d_model, d_model) * 0.1
final = concat @ Wo
print("Final output shape:", final.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
tokens = ["Token 1", "Token 2", "Token 3", "Token 4"]

sns.heatmap(A1, annot=True, xticklabels=tokens, yticklabels=tokens, cmap="viridis", ax=axes[0], cbar=True)
axes[0].set_title("Attention Weights - Head 1")
axes[0].set_xlabel("Keys")
axes[0].set_ylabel("Queries")

sns.heatmap(A2, annot=True, xticklabels=tokens, yticklabels=tokens, cmap="viridis", ax=axes[1], cbar=True)
axes[1].set_title("Attention Weights - Head 2")
axes[1].set_xlabel("Keys")
axes[1].set_ylabel("Queries")

plt.tight_layout()
plt.show()

### Analysis

#### 1. Why does each head potentially learn to attend to different relationships?
Each attention head has its own set of projection matrices ($W_q$, $W_k$, $W_v$) initialized randomly. During training, backpropagation updates these weights according to the loss signal. Because they start in different locations of the weight space, they optimize to capture different aspects of the sequence (different representation subspaces). For example, one head might focus on syntactic dependencies (e.g., subject-verb pairings), another on positional patterns (e.g., the next token), and another on semantic similarities.

#### 2. What would happen if all heads used identical weight matrices?
If all heads used identical weight matrices, they would project the input $X$ into the exact same query, key, and value spaces. The resulting attention weight matrices and output representations of each head would be identical. Concatenating them would just replicate the same representation multiple times, collapsing the multi-head mechanism into a single-head attention mechanism. This redundancy eliminates the model's ability to focus on different information subspaces simultaneously.